# DevPilot AI — CodeBERT Defect Classification Fine-Tuning
### Google Colab GPU Acceleration Notebook

This notebook fine-tunes **Microsoft CodeBERT** (`microsoft/codebert-base`) on the **CodeXGLUE Defect Detection** benchmark.

- **Task:** Binary function-level defect classification ($y \in \{0: \text{Clean}, 1: \text{Vulnerable/Buggy}\}$).
- **Hardware:** Free Google Colab T4 / V100 GPU (`Runtime` -> `Change runtime type` -> `T4 GPU`).
- **Output:** Model checkpoint (`best_model.pt`) for direct deployment into DevPilot AI.

In [ ]:
# 1. Install Dependencies
!pip install -q transformers torch datasets evaluate scikit-learn accelerate matplotlib

In [ ]:
# 2. Verify GPU Acceleration
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected. Switch Colab runtime to T4 GPU for fast training.")

In [ ]:
# 3. Load Microsoft CodeXGLUE Defect Detection Benchmark
from datasets import load_dataset

print("Downloading CodeXGLUE Defect Detection dataset...")
dataset = load_dataset("code_x_glue_cc_defect_detection")
print("Train samples:", len(dataset["train"]))
print("Validation samples:", len(dataset["validation"]))
print("Test samples:", len(dataset["test"]))

In [ ]:
# 4. Tokenization with CodeBERT Tokenizer
from transformers import AutoTokenizer

MODEL_NAME = "microsoft/codebert-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess_function(examples):
    return tokenizer(
        examples["func"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

tokenized_dataset = dataset.map(preprocess_function, batched=True, batch_size=500)

In [ ]:
# 5. Define Evaluation Metrics (Accuracy, Precision, Recall, F1)
import numpy as np
import evaluate
from sklearn.metrics import classification_report, confusion_matrix

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=predictions, references=labels)["f1"]
    prec = precision_metric.compute(predictions=predictions, references=labels)["precision"]
    rec = recall_metric.compute(predictions=predictions, references=labels)["recall"]
    return {"accuracy": acc, "f1": f1, "precision": prec, "recall": rec}

In [ ]:
# 6. Initialize CodeBERT Sequence Classifier
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

training_args = TrainingArguments(
    output_dir="./codebert_checkpoints",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

In [ ]:
# 7. Train CodeBERT
print("Starting CodeBERT Fine-Tuning...")
trainer.train()

In [ ]:
# 8. Test Set Evaluation & Confusion Matrix
test_results = trainer.predict(tokenized_dataset["test"])
test_preds = np.argmax(test_results.predictions, axis=-1)
test_labels = test_results.label_ids

print("\n--- Test Set Classification Report ---")
print(classification_report(test_labels, test_preds, target_names=["Clean (0)", "Defective (1)"]))

cm = confusion_matrix(test_labels, test_preds)
print("Confusion Matrix:\n", cm)

In [ ]:
# 9. Export Checkpoint for DevPilot AI Platform
import os
import json

export_dir = "./devpilot_codebert_export"
os.makedirs(export_dir, exist_ok=True)

trainer.save_model(export_dir)
tokenizer.save_pretrained(export_dir)

metrics_dict = {
    "model_name": "microsoft/codebert-base",
    "test_accuracy": float(accuracy_metric.compute(predictions=test_preds, references=test_labels)["accuracy"]),
    "test_f1": float(f1_metric.compute(predictions=test_preds, references=test_labels)["f1"]),
    "test_precision": float(precision_metric.compute(predictions=test_preds, references=test_labels)["precision"]),
    "test_recall": float(recall_metric.compute(predictions=test_preds, references=test_labels)["recall"]),
    "confusion_matrix": {
        "tn": int(cm[0][0]),
        "fp": int(cm[0][1]),
        "fn": int(cm[1][0]),
        "tp": int(cm[1][1])
    }
}

with open(os.path.join(export_dir, "model_metadata.json"), "w") as f:
    json.dump(metrics_dict, f, indent=2)

print("Successfully exported model artifacts to:", export_dir)
print("Download this folder and place inside ai_engine/deep_learning/checkpoints/ in DevPilot AI.")